# Session 8 · Grouping and reshaping

Work top to bottom and run each cell with **Shift+Enter**.

Cells under a **Your turn** heading contain `...` placeholders. Replace them, run the cell, then run the check cell beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook.

The data is the **synthetic** 2025 season from Session 3 (`data/matches_2025_synthetic.csv`): invented results for practice only. Solutions are in the matching `_solutions` notebook.

In [38]:
def check(label, get, expected):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [39]:
import numpy as np
import pandas as pd

matches = pd.read_csv("../data/matches_2025_synthetic.csv", parse_dates=["date"])
matches = matches.assign(
    home_score=lambda d: d["home_goals"] * 6 + d["home_behinds"],
    away_score=lambda d: d["away_goals"] * 6 + d["away_behinds"],
    total_goals=lambda d: d["home_goals"] + d["away_goals"],
    margin=lambda d: d["home_score"] - d["away_score"],
)
matches.head(3)

,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd,home_score,away_score,total_goals,margin
0,1,2025,1,2025-03-16,Collingwood,Carlton,12,12,19,11,14893.0,84,125,31,-41
1,2,2025,1,2025-03-16,Hawthorn,Brisbane,14,9,17,10,12927.0,93,112,31,-19
2,3,2025,1,2025-03-15,North Melbourne,Richmond,16,11,10,9,21940.0,107,69,26,38


## 1. `groupby` with named aggregation: `group_by` + `summarise`

```r
matches %>% group_by(round) %>% summarise(games = n(), avg_goals = mean(total_goals))
```

becomes

```python
matches.groupby("round", as_index=False).agg(
    games=("match_id", "count"),
    avg_goals=("total_goals", "mean"),
)
```

Each new column is `name=(source_column, function)`. `as_index=False` keeps `round` as an ordinary column, like an ungrouped tibble. Without it, `round` becomes the row index.

In [40]:
matches.groupby("home_team", as_index=False).agg(
    home_games=("match_id", "count"),
    avg_home_score=("home_score", "mean"),
).sort_values("avg_home_score", ascending=False).head()

,home_team,home_games,avg_home_score
11,North Melbourne,12,104.416667
0,Adelaide,9,99.333333
16,West Coast,9,94.444444
12,Port Adelaide,11,91.909091
2,Carlton,12,91.500000


### Your turn 1.1

Build `by_round` with one row per round and these columns:

- `games`: the number of matches
- `avg_total_goals`: the mean of `total_goals`
- `max_crowd`: the largest crowd

In [41]:
by_round = matches.groupby("round", as_index=False).agg(
    games=("match_id", "count"),
    avg_total_goals=("total_goals", "mean"),
    max_crowd=("crowd", "max")
)

In [42]:
check("1.1 rows", lambda: len(by_round), 23)
check("1.1 round 1 avg_total_goals", lambda: by_round.loc[by_round["round"] == 1, "avg_total_goals"].iloc[0], 23.88888888888889)
check("1.1 round 1 max_crowd", lambda: float(by_round.loc[by_round["round"] == 1, "max_crowd"].iloc[0]), 64499.0)

PASS     1.1 rows
PASS     1.1 round 1 avg_total_goals
PASS     1.1 round 1 max_crowd


### Your turn 1.2

`.size()` counts rows per group, like `count()`. Filter to home wins (`margin > 0`), then count them per home team. Set `geelong_home_wins` to Geelong's count.

Hint: the result is a Series indexed by team, so `.get("Geelong", 0)` reads one value safely.

In [43]:
home_wins = matches.query("margin > 0").groupby("home_team", as_index=False).size()
geelong_home_wins = matches.query("margin > 0 and home_team == 'Geelong'").shape[0]

In [44]:
check("1.2 geelong_home_wins", lambda: geelong_home_wins, 7)

PASS     1.2 geelong_home_wins


## 2. `transform`: `group_by` + `mutate`

`agg` collapses each group to one row. `transform` returns a value for **every** row, broadcast back to the original shape, which is what `group_by() %>% mutate()` does in dplyr.

In [45]:
matches["round_avg_margin"] = matches.groupby("round")["margin"].transform("mean")
matches[["round", "home_team", "margin", "round_avg_margin"]].head()

,round,home_team,margin,round_avg_margin
0,1,Collingwood,-41,-3.333333
1,1,Hawthorn,-19,-3.333333
2,1,North Melbourne,38,-3.333333
3,1,Essendon,-10,-3.333333
4,1,GWS,-59,-3.333333


### Your turn 2.1

Add `crowd_share`: each match's crowd as a share of its round's total crowd. Use `transform("sum")` for the denominator. Missing crowds are skipped in the sum, so each round's shares add up to 1.

In [46]:
matches["crowd_share"] = matches["crowd"] / matches.groupby("round")["crowd"].transform("sum")

In [47]:
check("2.1 round 1 shares sum to 1", lambda: round(matches.loc[matches["round"] == 1, "crowd_share"].sum(), 6), 1.0)

PASS     2.1 round 1 shares sum to 1


## 3. Reshaping: `pivot_longer` and `pivot_wider`

| tidyr | pandas |
| --- | --- |
| `pivot_longer(cols = c(a, b), names_to = "side", values_to = "team")` | `df.melt(id_vars=[...], value_vars=["a", "b"], var_name="side", value_name="team")` |
| `pivot_wider(names_from = phase, values_from = x)` | `df.pivot(index=..., columns="phase", values="x")` |
| `pivot_wider` with duplicates summarised | `df.pivot_table(index=..., columns=..., values=..., aggfunc="mean")` |

Use `pivot` when each cell should hold exactly one value: it raises an error on duplicates, which catches mistakes. `pivot_table` silently aggregates them.

### Your turn 3.1

Every team plays once per round. Prove it: melt `home_team` and `away_team` into one `team` column, keeping `match_id` and `round` as identifiers. Then count games per team.

- `teams_long`: the melted table (it should have 414 rows: two per match)
- `games_per_team`: `teams_long["team"].value_counts()`

In [48]:
teams_long = matches.melt(
    id_vars=["match_id", "round"],
    value_vars=["home_team", "away_team"],
    var_name="home_or_away",
    value_name="team"
)
games_per_team = teams_long["team"].value_counts()

In [49]:
check("3.1 rows", lambda: len(teams_long), 414)
check("3.1 every team played 23", lambda: set(games_per_team), {23})

PASS     3.1 rows
PASS     3.1 every team played 23


### Your turn 3.2

This is the same reshape as your Milestone 1 `season_gap` table, one level down:

1. Add a `phase` column: `"Early"` for rounds 6 and under, otherwise `"Rest"`
2. `wide`: one row per home team, with `Early` and `Rest` columns holding the mean home score (`pivot_table`, then `.reset_index()`)
3. `long_again`: melt `wide` back to long form, with columns `home_team`, `phase` and `avg_home_score`

In [50]:
matches["phase"] = matches["round"].apply(lambda r: "Early" if r <= 6 else "Rest")
wide = matches.pivot_table(index="home_team", columns="phase", values="home_score", aggfunc="mean").reset_index()
long_again = wide.melt(id_vars="home_team", var_name="phase", value_name="avg_home_score")

In [51]:
check("3.2 Geelong early", lambda: round(float(wide.loc[wide["home_team"] == "Geelong", "Early"].iloc[0]), 6), 87.5)
check("3.2 long_again shape", lambda: long_again.shape, (36, 3))

PASS     3.2 Geelong early
PASS     3.2 long_again shape


## Wrap-up

Run All, confirm every check passes, and commit from VS Code's Source Control panel with the message `Session 8: grouping and reshaping`.